# SPK-3 — Driver OOM (collecting unbounded data to the driver)

**Break → Detect → Fix → Prove.** The driver is a single JVM. Pulling a large result back to it
with `.collect()` / `.toPandas()` (or an oversized `F.broadcast`) exhausts its heap — the job
dies and, over **Spark Connect**, the whole session dies with it.

**Pre-requisite:** `make up` → open the Spark UI at http://localhost:4040.

**Laptop-safety / why we *describe* the crash:** a genuine driver-heap OOM over Spark Connect
**hard-kills the session** — every later cell then raises `[NO_ACTIVE_SESSION]` and a re-import
won't help. So we **describe** the antipattern (and how to recover) and **execute the safe
patterns** you should use instead. To see the crash for real, run the snippet below on
`make up-constrained` (1 GB driver) and recover with `reconnect()`.

In [ ]:
from common.spark_session import spark, display_df, reconnect
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, wide_rows, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & the generated-large frame

A **wide** fact (many columns → large bytes per row) is the "too big to collect" frame; a small
dimension is safe to broadcast.

In [ ]:
N_ROWS  = 5_000_000
N_COLS  = 20
N_KEYS  = 2_000

wide_fact = wide_rows(spark, n_rows=N_ROWS, n_cols=N_COLS)                 # "too big to collect"
fact      = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS, key_col="customer_id")
dim       = key_dimension(spark, n_keys=N_KEYS, key_col="customer_id")     # small -> safe to broadcast

def _safe_conf(k):
    try: return spark.conf.get(k)            # no string default -> never hits the byte parser
    except Exception: return "<unset>"
print(f"wide_fact: ~{N_ROWS:,} rows x {N_COLS} cols (lazy)   dim: {N_KEYS:,} rows")
print(f"driver.maxResultSize = {_safe_conf('spark.driver.maxResultSize')}   "
      f"driver.memory = {_safe_conf('spark.driver.memory')}")
wide_fact.limit(3).toPandas()                                              # safe: only 3 rows

## 1. Break it — the antipattern (described, not run)

```python
wide_fact.collect()          # pulls ALL ~5M x 20 doubles to the driver JVM
# or: wide_fact.toPandas()   # same thing
# or: fact.join(F.broadcast(wide_fact), ...)   # broadcasting a huge frame = same OOM
```

Each pulls the **entire** frame into the driver's heap. The driver OOMs; over Spark Connect the
session is **killed** (not a clean exception you can catch — pyspark's own error path then also
raises `[NO_ACTIVE_SESSION]`). Running it here would abort the rest of the notebook, so we don't —
the point is **never materialize unbounded data on the driver.**

## Recovery — if you *do* kill the session

```python
spark = reconnect()          # fresh Spark Connect session after a driver death
```

> **Gotcha:** `reconnect()` gives a *new* session, but DataFrames built before the death are
> bound to the *old* one and still raise `[NO_ACTIVE_SESSION]` — rebuild them (re-run the setup
> cells) after reconnecting.

## 2. Detect it — read the Spark UI

http://localhost:4040 → **Executors** tab: watch the **driver**'s Storage/On-Heap memory climb as
a collect runs; a driver heap OOM shows as the app/session dying (see
[`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)). The confs that bound this:

In [ ]:
profile_summary(spark, keys=["spark.driver.maxResultSize", "spark.driver.memory",
                              "spark.sql.autoBroadcastJoinThreshold"])

## 3. Fix it (A) — aggregate on the cluster, return a tiny result

Do the heavy work as a distributed aggregation; only a handful of summary rows reach the driver.

In [ ]:
agg = (fact.groupBy("customer_id")
            .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue")))
m_agg = measure(spark, "aggregate (cluster-side)", lambda: agg.count())
print("aggregate metrics:", m_agg)
agg.orderBy(F.desc("revenue")).limit(5).toPandas()   # tiny, safe to pull

## 3. Fix it (B) — `limit()` before collecting (the right way to peek)

In [ ]:
SAMPLE = 1_000
m_limit = measure(spark, "limit -> pandas", lambda: wide_fact.limit(SAMPLE).toPandas())
print(f"limit({SAMPLE}) -> pandas metrics:", m_limit)

## 3. Fix it (C) — broadcast only the *small* side

In [ ]:
apply_profile(spark, "tuned")   # broadcast enabled (10 MB threshold)
good_broadcast = fact.join(F.broadcast(dim), on="customer_id", how="inner")   # small side only
m_bcast = measure(spark, "broadcast small dim", lambda: good_broadcast.count())
print("broadcast (small side) metrics:", m_bcast)

## 4. Prove it

The safe patterns all keep the heavy work on the cluster and return tiny results to the driver —
fast, and the session stays alive. The unbounded collect (not run) would never return at all.

In [ ]:
m_broken = {"label": "collect unbounded (NOT run — would OOM the driver)", "runtime_s": None}
compare([m_broken, m_agg, m_limit, m_bcast])
print("\nKeep the heavy work on the cluster; only small, bounded results should cross to the driver.")

## Takeaways & "in real production…"

- **Never collect unbounded data to the driver** — aggregate/`limit` on the cluster; write to a
  table instead of collecting; broadcast only genuinely small sides.
- Over **Spark Connect**, a driver OOM **kills the session** (not a catchable error) — recover with
  `reconnect()` and rebuild your DataFrames.
- In prod: set `spark.driver.maxResultSize` deliberately, alert on driver memory, and review jobs
  for `.collect()` / `.toPandas()` on unbounded inputs.

## Teardown

Nothing was written, and the destructive collect was never run. Reset the profile.

In [ ]:
apply_profile(spark, "tuned")
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'.")